# myMIA — my Medical Insurance Agent

Annual enrollment advisor. Everything year-specific comes from one input: the **annual profile JSON**.

```
medicalAgent profile ──► annualProfile.json ──► claims + plans ──► scenarios ──► ranking ──► 10-yr story
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))

import logging
import pandas as pd
import matplotlib.pyplot as plt
from medicalAgent.myMIA import load_annual, load_claims, price_all, rank, outlook, plan_table
from medicalAgent.myMIA.claims import annualize_factor

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
pd.options.display.float_format = '{:,.0f}'.format

## Input
- `YEAR` → `<medInsDir>/<YEAR>/annualProfile.json` (medInsDir comes from the medicalAgent profile)
- or set `ANNUAL_PROFILE` to an explicit path

In [ ]:
YEAR = 2027
ANNUAL_PROFILE = None

prof, ann = load_annual(year=YEAR, path=ANNUAL_PROFILE)
print(f"{prof['personId']} | {prof['insurer']} | enrolling for {ann['year']} | {ann['_path']}")

## 1. Last year's claims (my history)

In [ ]:
df = load_claims(ann)
f = annualize_factor(df, ann)
print(f"{len(df)} claims, {df.Date.min().date()} → {df.Date.max().date()}, annualize x{f:.2f}, duplicates dropped: {df.attrs['duplicatesDropped']}")
spend = df.groupby('medDetail').agg(claims=('cost','count'), billed=('cost','sum'), myshare=('myshare','sum')).sort_values('myshare', ascending=False)
spend.loc['TOTAL'] = spend.sum()
spend

## 2. The plans and what changed

In [ ]:
print(*['• ' + c for c in ann['yoyChanges']], sep='\n')
plan_table(ann)

## 3. Scenarios — what my year costs under each plan
Premium + medical OOP (capped at plan max) + pharmacy OOP. Healthy → baseline (replay of my claims) → inflated → illness (2 ER + 1 inpatient stay).

In [ ]:
priced = price_all(ann, df)
sc = list(ann['scenarios'])
priced.pivot(index='scenario', columns='plan', values='total').loc[sc]

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
w = 0.38
for i, pn in enumerate(ann['plans']):
    d = priced[priced.plan==pn].set_index('scenario').loc[sc]
    x = [j + (i-0.5)*w for j in range(len(sc))]
    b = 0
    for col, hatch, a in (('premium','',0.95),('medical','//',0.65),('pharma','..',0.4)):
        ax.bar(x, d[col], w, bottom=b, label=f'{pn} {col}', hatch=hatch, color=f'C{i}', alpha=a, edgecolor='white')
        b = b + d[col].values
ax.set_xticks(range(len(sc))); ax.set_xticklabels(sc); ax.set_ylabel('USD / year'); ax.legend(fontsize=7, ncol=2)
ax.set_title(f"{ann['year']} annual cost by scenario"); plt.show()

## 4. Ranking
Expected = scenario-weighted (weights live in the annual profile). WORST = highest scenario cost.

In [ ]:
r = rank(ann, priced)
print({k: s['weight'] for k, s in ann['scenarios'].items()})
r

In [ ]:
exp = r.loc['EXPECTED'].sort_values()
worst = r.loc['WORST']
best, other = exp.index[0], exp.index[1]
print(f"RECOMMEND: {best}")
print(f"  expected saving vs {other}: ${exp[other]-exp[best]:,.0f}/yr")
print(f"  worst case: {best} ${worst[best]:,.0f} vs {other} ${worst[other]:,.0f}")
wins = r.drop(['EXPECTED','WORST']).idxmin(axis=1)
print('  cheapest per scenario:', ', '.join(f'{k}→{v}' for k, v in wins.items()))

## 5. The next 5 and 10 years — my story
Premiums, medical prices and my own usage all grow (rates in `projection`). Plan caps held flat — conservative for the high-cap plan.

In [ ]:
o = outlook(ann, priced)
fig, axs = plt.subplots(1, 2, figsize=(11,4))
for ax, col, t in ((axs[0],'expected','Expected mix'), (axs[1],'illness','Illness every year')):
    for pn in ann['plans']:
        d = o[o.plan==pn]; ax.plot(d.year, d[col], marker='o', label=pn)
    ax.set_title(f'Cumulative cost — {t}'); ax.set_xlabel('years from now'); ax.set_ylabel('USD'); ax.legend()
plt.show()
o[o.year.isin([1,5,10])].pivot(index='year', columns='plan', values=['expected','illness'])

## 6. HRA runway

In [ ]:
wts = pd.Series({k: s['weight'] for k, s in ann['scenarios'].items()}, name='w')
x = priced.merge(wts, left_on='scenario', right_index=True)
x['oop'] = (x.medical + x.pharma) * x.w
oop = x.groupby('plan').oop.sum()
hra = ann['hraBalance']
print(f"HRA balance ${hra:,.0f}")
for pn, v in oop.items(): print(f"  {pn}: expected OOP ${v:,.0f}/yr → HRA covers ~{hra/v:.1f} yrs of OOP (before growth)")

## Notes

In [ ]:
print(*['• ' + n for n in ann['notes']], sep='\n')